In [18]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [19]:
raw = spark.read.csv("hdfs://namenode:9000/data/raw/accidentes/", header=True, inferSchema=True)
raw.write.mode("overwrite").parquet("hdfs://namenode:9000/data/staging/accidentes")

In [20]:
raw.count()

7728394

In [41]:
stg = spark.read.parquet("hdfs://namenode:9000/data/staging/accidentes/")

In [42]:
stg.count()

7728394

In [43]:
stg.columns

['ID',
 'Source',
 'Severity',
 'Start_Time',
 'End_Time',
 'Start_Lat',
 'Start_Lng',
 'End_Lat',
 'End_Lng',
 'Distance(mi)',
 'Description',
 'Street',
 'City',
 'County',
 'State',
 'Zipcode',
 'Country',
 'Timezone',
 'Airport_Code',
 'Weather_Timestamp',
 'Temperature(F)',
 'Wind_Chill(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Direction',
 'Wind_Speed(mph)',
 'Precipitation(in)',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Turning_Loop',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight']

In [44]:
from pyspark.sql import functions as F

df_counts = stg.groupBy("ID").count().filter("count > 1")
print("Cantidad de IDs que tienen duplicados:", df_counts.count())

df_counts.select("ID").show(10, truncate=False)

Cantidad de IDs que tienen duplicados: 0
+---+
|ID |
+---+
+---+



In [45]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType
exprs = []
total_filas = stg.count()
cols = [c for c in stg.columns if c != "ID"]
tipos = dict(stg.dtypes)
valores_nulos_texto = ["", "NaN", "nan", "None", "N/A", "NULL"]
for col_nombre in cols:
    if tipos[col_nombre] in ("double", "float"):
        condicion = F.col(col_nombre).isNull()| F.isnan(F.col(col_nombre))
    elif tipos[col_nombre] in ("timestamp", "boolean"):
        condicion = F.col(col_nombre).isNull()
    else: 
        condicion = F.trim(F.col(col_nombre)).isin(valores_nulos_texto) | F.col(col_nombre).isNull()
    exprs.append(F.sum(F.when(condicion, 1).otherwise(0)).alias(col_nombre))
conteos = stg.agg(*exprs).collect()[0].asDict()

In [46]:
for c in cols:
    n = conteos[c] or 0
    print(f"Columna: {c} | Nulos/NaN: {n} | Porcentaje: {n / total_filas * 100:.2f}%")

Columna: Source | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Severity | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: End_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Lat | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Lng | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: End_Lat | Nulos/NaN: 3402762 | Porcentaje: 44.03%
Columna: End_Lng | Nulos/NaN: 3402762 | Porcentaje: 44.03%
Columna: Distance(mi) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Description | Nulos/NaN: 5 | Porcentaje: 0.00%
Columna: Street | Nulos/NaN: 10869 | Porcentaje: 0.14%
Columna: City | Nulos/NaN: 253 | Porcentaje: 0.00%
Columna: County | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: State | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Zipcode | Nulos/NaN: 1915 | Porcentaje: 0.02%
Columna: Country | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Timezone | Nulos/NaN: 7808 | Porcentaje: 0.10%
Columna: Airport_Code | Nulos/NaN: 22635 | Porcentaje: 0.29%
Columna: Weather_T

In [47]:
stg= stg.drop("End_Lat", "End_Lng","Airport_Code","Zipcode","Country")

In [48]:
total_antes = stg.count()

stg = stg.dropna()

total_despues = stg.count()
total_eliminados = total_antes - total_despues

print(f"Registros originales: {total_antes:,}")
print(f"Registros limpios finales en accidents: {total_despues:,}")
print(f"Total de filas eliminadas: {total_eliminados:,}")

Registros originales: 7,728,394
Registros limpios finales en accidents: 5,208,003
Total de filas eliminadas: 2,520,391


In [49]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType
exprs = []
total_filas = stg.count()
cols = [c for c in stg.columns if c != "ID"]
tipos = dict(stg.dtypes)
valores_nulos_texto = ["", "NaN", "nan", "None", "N/A", "NULL"]
for col_nombre in cols:
    if tipos[col_nombre] in ("double", "float"):
        condicion = F.col(col_nombre).isNull()| F.isnan(F.col(col_nombre))
    elif tipos[col_nombre] in ("timestamp", "boolean"):
        condicion = F.col(col_nombre).isNull()
    else: 
        condicion = F.trim(F.col(col_nombre)).isin(valores_nulos_texto) | F.col(col_nombre).isNull()
    exprs.append(F.sum(F.when(condicion, 1).otherwise(0)).alias(col_nombre))
conteos = stg.agg(*exprs).collect()[0].asDict()

In [51]:
for c in cols:
    n = conteos[c] or 0
    print(f"Columna: {c} | Nulos/NaN: {n} | Porcentaje: {n / total_filas * 100:.2f}%")

Columna: Source | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Severity | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: End_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Lat | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Lng | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Distance(mi) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Description | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Street | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: City | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: County | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: State | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Timezone | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Weather_Timestamp | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Temperature(F) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Wind_Chill(F) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Humidity(%) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Pressure(in) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Visibility(m

In [ ]:
excluir = ["End_Lat", "End_Lng"]

cols_accidentes = [c for c in (stg.columns[0:19] + stg.columns[29:]) if c not in excluir]

accidents = stg.select(cols_accidentes)
accidents.show(2)

+---------+-------+--------+-------------------+-------------------+---------+-----------+------------+--------------------+---------+-----------+---------+-----+----------+-------+----------+------------+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|       ID| Source|Severity|         Start_Time|           End_Time|Start_Lat|  Start_Lng|Distance(mi)|         Description|   Street|       City|   County|State|   Zipcode|Country|  Timezone|Airport_Code|Crossing|Give_Way|Junction|No_Exit|Railway|Roundabout|Station| Stop|Traffic_Calming|Traffic_Signal|Turning_Loop|Sunrise_Sunset|Civil_Twilight|Nautical_Twilight|Astronomical_Twilight|
+---------+-------+--------+-------------------+-------------------+---------+-----------+------------+--------------------+---------+-----------+---------+-----+----------+-------+----------+------------+--------+--------+-

In [ ]:
weather = stg.select("ID",*stg.columns[19:29])

weather.show(2)

+---------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+--------------------+-------+
|       ID|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|   Weather_Condition|Amenity|
+---------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+--------------------+-------+
|A-5199312|2022-10-07 23:54:00|          79.0|         79.0|       34.0|        29.5|           8.0|             E|            5.0|              0.0|                Fair|  false|
|A-5199313|2023-01-28 03:15:00|          32.0|         19.0|       64.0|       26.21|          10.0|             W|           23.0|              0.0|Partly Cloudy / W...|  false|
+---------+-------------------+--------------+-------------+-----------+------------+--------------+-----